# Module 11 - 실습 1 솔루션: 검증 피라미드 구현

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import re
from typing import Type
from enum import Enum
from dataclasses import dataclass
from pydantic import BaseModel, Field, field_validator, ValidationError
from common.fake_llm import FakeLLM

print("검증 피라미드 솔루션 준비 완료")

## 실습 1 솔루션: Layer 0 - JSON 파싱

In [ ]:
def parse_json_from_llm(text: str) -> dict:
    """LLM 출력에서 JSON을 추출합니다."""
    # 1단계: 직접 파싱
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # 2단계: 마크다운 JSON 블록 추출
    pattern = r"```(?:json)?\s*\n?(.*?)\n?\s*```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    
    # 3단계: 중괄호 범위 추출
    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace != -1 and last_brace > first_brace:
        try:
            return json.loads(text[first_brace:last_brace + 1])
        except json.JSONDecodeError:
            pass
    
    raise ValueError(f"JSON 파싱 실패: {text[:200]}...")

In [ ]:
# 검증
result1 = parse_json_from_llm('{"key": "value", "num": 42}')
assert result1 == {"key": "value", "num": 42}

result2 = parse_json_from_llm('다음은 결과입니다:\n```json\n{"summary": "분석 완료", "confidence": 0.85}\n```')
assert result2["summary"] == "분석 완료"

result3 = parse_json_from_llm('분석 결과: {"status": "ok", "score": 0.9} 이상입니다.')
assert result3["status"] == "ok"

print("✅ 실습 1 완료! parse_json_from_llm이 3가지 형태를 모두 처리합니다.")

## 실습 2 솔루션: Layer 1 - Pydantic 스키마

In [ ]:
class AnalysisResult(BaseModel):
    """LLM 분석 결과 스키마."""
    summary: str = Field(min_length=1, description="분석 요약")
    confidence: float = Field(ge=0.0, le=1.0, description="분석 신뢰도")
    issues: list[str] = Field(default_factory=list, description="발견된 문제")
    recommendation: str = Field(min_length=1, description="권장 조치")
    
    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        """confidence를 소수점 2자리로 반올림합니다."""
        return round(v, 2)

In [ ]:
# 검증
valid_data = {
    "summary": "로그인 API에 인증 누락 취약점이 발견되었습니다.",
    "confidence": 0.856,
    "issues": ["토큰 검증 미수행"],
    "recommendation": "JWT 검증 미들웨어를 추가하세요."
}
result = AnalysisResult.model_validate(valid_data)
assert result.confidence == 0.86
assert result.summary == valid_data["summary"]

partial_data = {"summary": "분석 완료", "confidence": 0.9, "recommendation": "조치 필요"}
result2 = AnalysisResult.model_validate(partial_data)
assert result2.issues == []

print(f"confidence 반올림: 0.856 → {result.confidence}")
print("✅ 실습 2 완료! AnalysisResult 스키마가 올바르게 검증합니다.")

## 실습 3 솔루션: Layer 2 - Confidence 게이팅

In [ ]:
class GateAction(str, Enum):
    ACCEPT = "accept"
    RETRY = "retry"
    REJECT = "reject"

@dataclass
class GateResult:
    action: GateAction
    confidence: float
    message: str

def confidence_gate(
    confidence: float,
    threshold_accept: float = 0.7,
    threshold_reject: float = 0.4,
) -> GateResult:
    """Confidence 값에 따라 ACCEPT / RETRY / REJECT를 결정합니다."""
    if confidence >= threshold_accept:
        action = GateAction.ACCEPT
        message = f"Confidence {confidence:.2f} >= {threshold_accept} → 수락"
    elif confidence >= threshold_reject:
        action = GateAction.RETRY
        message = f"Confidence {confidence:.2f} — 재시도 권장"
    else:
        action = GateAction.REJECT
        message = f"Confidence {confidence:.2f} < {threshold_reject} → 거부"
    
    return GateResult(action=action, confidence=confidence, message=message)

In [ ]:
# 검증
assert confidence_gate(0.85).action == GateAction.ACCEPT
assert confidence_gate(0.55).action == GateAction.RETRY
assert confidence_gate(0.25).action == GateAction.REJECT
assert confidence_gate(0.7).action == GateAction.ACCEPT
assert confidence_gate(0.4).action == GateAction.RETRY

print(f"0.85 → {confidence_gate(0.85).action.value}")
print(f"0.55 → {confidence_gate(0.55).action.value}")
print(f"0.25 → {confidence_gate(0.25).action.value}")
print("✅ 실습 3 완료! confidence_gate가 올바르게 분류합니다.")

## 실습 4 솔루션: 전체 파이프라인

In [ ]:
def parse_and_validate(text: str) -> dict:
    """LLM 출력 텍스트를 파싱하고 검증합니다."""
    try:
        # Layer 0: JSON 파싱
        raw = parse_json_from_llm(text)
        
        # Layer 1: 스키마 검증
        validated = AnalysisResult.model_validate(raw)
        result_dict = validated.model_dump()
        
        # Layer 2: Confidence 게이팅
        gate = confidence_gate(result_dict["confidence"])
        
        return {"result": result_dict, "gate": gate}
    
    except Exception as e:
        return {"error": str(e)}

In [ ]:
fake_llm = FakeLLM(responses={
    "분석": '{"summary": "인증 누락 취약점", "confidence": 0.85, "issues": ["JWT 미검증"], "recommendation": "미들웨어 추가"}',
})

llm_output_high = fake_llm.invoke("코드를 분석해주세요").content
result_high = parse_and_validate(llm_output_high)
print(f"높은 신뢰도 결과: gate={result_high.get('gate', {})}, action={result_high.get('gate', {}).action if 'gate' in result_high else 'N/A'}")

low_confidence_json = '{"summary": "불확실", "confidence": 0.2, "issues": [], "recommendation": "재검토"}'
result_low = parse_and_validate(low_confidence_json)
print(f"낮은 신뢰도 결과: action={result_low.get('gate', {}).action if 'gate' in result_low else 'N/A'}")

In [ ]:
# 검증
assert result_high is not None
if "gate" in result_high:
    assert result_high["gate"].action in [GateAction.ACCEPT, GateAction.RETRY]

if "gate" in result_low:
    assert result_low["gate"].action == GateAction.REJECT

print("✅ 실습 4 완료! 전체 검증 파이프라인이 올바르게 작동합니다.")